In [ ]:
from gould_2026.sim_stim import run_sim_stim
from gould_2026.datasets import Naumann24uDataset
from gould_2026.estimator import Pipeline, ArrayWithTime
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)

In [ ]:
d = Naumann24uDataset(1)
d.neural_data[np.isnan(d.neural_data)] = 0

In [ ]:
sr, stim_designer, log = run_sim_stim(
    d.neural_data,
    rng=rng,
    behavioral_data=d.behavioral_data,
    initial_nostim_period=np.inf,
    show_tqdm=True,
    exit_time=np.inf,
    prosvd_k=10,
    beh_decay_rate=0,
)

In [ ]:
latents = log['latents']

def beh_S(point, bottom=-1.24, top=2.4):
    point = point / 8
    quadratic = point[0] ** 2 - 2 * point[1] ** 2
    surface = np.tanh(quadratic) * (top - bottom) / 2
    surface = surface - (-(top - bottom) / 2 - bottom)
    if np.isnan(surface):
        return 0
    else:
        return surface


c = ArrayWithTime([beh_S(point) for point in latents], latents.t)

stim_c, stim_latents = (x.slice_by_time(slice(d.end_of_visual_period_time,None)) for x in [c, latents])

plt.scatter(latents[:,0], latents[:,1], c=c)

In [ ]:
d.end_of_visual_period_time

In [ ]:
%matplotlib qt
b = log['behavior']

fig, axs = plt.subplots(nrows=1, ncols=1, sharex=True, sharey=True, constrained_layout=True, figsize=(10,5), squeeze=False)
axs[0,0].plot(b.t, b)
axs[0,0].plot(d.behavioral_data.t, d.behavioral_data)

axs[0,0].set_xlim(d.end_of_visual_period_time, b.t[-1])

axs[0,0].legend(['artificial behavior based on neural state', 'actual behavior'])

# diff = ArrayWithTime.subtract_aligned_indices(d.behavioral_data, b)
# plt.plot(diff.t, diff)